# Linear Algebra Deconvolution Simulation

Andrew E. Davidson aedavids@ucsc.edu 1/29/25 

Copyright (c) 2020-2023, Regents of the University of California All rights reserved.   https://polyformproject.org/licenses/noncommercial/1.0.0


CIBERSORTx is unable to deconvolve the serial dilution samples. Try relaxing the count & non-negative constraints. Solve using standard linear algebra

ref: linearAlgebraDeconvolution.ipynb
linearAlgebraDeconvolution.ipynb did not work. try using simulated data

In [1]:
import ipynbname

import numpy as np
import os
import pandas as pd

In [2]:
notebookName = ipynbname.name()
notebookPath = ipynbname.path()
notebookDir = os.path.dirname(notebookPath)

outDir = f'{notebookDir}/{notebookName}.out'

dataOut = f'{outDir}/data'
os.makedirs(dataOut, exist_ok=True) 
print(f'dataOut:\n{dataOut}')

# imgOut = f'{outDir}/img'
# os.makedirs(imgOut, exist_ok=True) 
# print(f'imgOut:\n{imgOut}')


dataOut:
/private/home/aedavids/extraCellularRNA/deconvolutionAnalysis/python/tempus/jupyterNotebooks/linearAlgebraDeconvolutionSimulation.out/data


## Generate Random Data

In [3]:
meaningOfLife = 42
np.random.seed(meaningOfLife)

numGenes = 15

def creatSignatureDataFrame( numGenes: int ) -> (pd.DataFrame, list[str]):
    '''
    TODO
    '''

    # multiply by a million so we have a strong signal with similar order of magnitude to
    # /private/groups/kimlab/aedavids/deconvolution/tempus/best/bestArithemetic10/bestArithemetic10Tempus.sh.out/
    # cibersortInputDir/arithmeticBiomarkers_T1.csv
    
    undiluted,control = np.random.rand(2, numGenes) * 1e6
    
    # print( f'undiluted\n {undiluted}')
    # print( f'\ncontrol\n {control}')
    
    geneIds = ['g' + str(i + 1) for i in range(numGenes) ]
    geneIds
    
    signatureDF = pd.DataFrame(
            data = np.array( [undiluted, control] ).transpose(),
            index = geneIds,
            columns = ['undiluted', 'control']
    )
    signatureDF.index.name = 'gene_id'

    return (signatureDF, geneIds)


signatureDF, geneIds = creatSignatureDataFrame( numGenes )
print(f'geneIds : {geneIds}')
print(f'signatureDF\n')
signatureDF

geneIds : ['g1', 'g2', 'g3', 'g4', 'g5', 'g6', 'g7', 'g8', 'g9', 'g10', 'g11', 'g12', 'g13', 'g14', 'g15']
signatureDF



,undiluted,control
gene_id,,
g1,374540.118847,183404.509853
g2,950714.306410,304242.242960
g3,731993.941811,524756.431632
g4,598658.484197,431945.018642
g5,156018.640442,291229.140198
g6,155994.520336,611852.894722
g7,58083.612168,139493.860652
g8,866176.145775,292144.648535
g9,601115.011743,366361.843294


In [4]:
# create mixture matrix

def createMixtureDF( 
        signatureDF : pd.DataFrame,
        dilutions : list[float]
) -> pd.DataFrame :
    '''
    TODO
    '''
    undilutedSeries = signatureDF.loc[:, "undiluted"]
    controlSeries   = signatureDF.loc[:, "control"]

    retDF = signatureDF.copy()
    
    for dilution in dilutions :
        sampleId = f'inSilco_{dilution}'
        c = controlSeries * (dilution)
        d = undilutedSeries * (1.0 - dilution)
        s = c + d
        retDF[ sampleId ] = s

    return retDF

dilutions = [1e-1, 1e-2, 1e-3, 1e-4, 1e-5, 1e-6]
mixtureDF = createMixtureDF( signatureDF,  dilutions)

print(f'mixtureDF\n')
mixtureDF

mixtureDF



,undiluted,control,inSilco_0.1,inSilco_0.01,inSilco_0.001,inSilco_0.0001,inSilco_1e-05,inSilco_1e-06
gene_id,,,,,,,,
g1,374540.118847,183404.509853,355426.557948,372628.762757,374348.983238,374521.005286,374538.207491,374539.927712
g2,950714.306410,304242.242960,886067.100065,944249.585775,950067.834346,950649.659204,950707.841689,950713.659938
g3,731993.941811,524756.431632,711270.190793,729921.566710,731786.704301,731973.218060,731991.869436,731993.734574
g4,598658.484197,431945.018642,581987.137642,596991.349541,598491.770731,598641.812850,598656.817062,598658.317484
g5,156018.640442,291229.140198,169539.690418,157370.745440,156153.850942,156032.161492,156019.992547,156018.775653
g6,155994.520336,611852.894722,201580.357775,160553.104080,156450.378711,156040.106174,155999.078920,155994.976195
g7,58083.612168,139493.860652,66224.637017,58897.714653,58165.022417,58091.753193,58084.426271,58083.693578
g8,866176.145775,292144.648535,808772.996051,860435.830803,865602.114278,866118.742625,866170.405460,866175.571743
g9,601115.011743,366361.843294,577639.694898,598767.480059,600880.258575,601091.536426,601112.664212,601114.776990


In [5]:
def solveForFractions( 
    signatureDF : pd.DataFrame,
    mixtureDF : pd.DataFrame,
) -> np.array :
    '''
    TODO
    '''
    # select numeric cols
    numericCols = ~signatureDF.columns.isin( ["gene_id"] )                                     
    sigNP = signatureDF.loc[:, numericCols].values
    
    # select numeric columns
    numericCols = ~mixtureDF.columns.isin( ["gene_id"] )                                     
    mixtureNP = mixtureDF.loc[:, numericCols].values

    print(f' sigNP.shape : {sigNP.shape} mixtureNP.shape : {mixtureNP.shape}')
    # 
    # Calculate the pseudo-inverse of the signature matrix
    signaturePinvNP = np.linalg.pinv( sigNP )
    # print(f"\n signaturePinvNP.shape : {signaturePinvNP.shape}\n{signaturePinvNP}")
    
    print("\n\n")
    
    FTranspose = np.matmul( signaturePinvNP, mixtureNP)
    # print(f"Matrix FTranspose.shape :{FTranspose.shape}")
    # print(FTranspose)
    
    # print(f"\n\nF.shape :{FTranspose.shape}")
    # print(np.transpose( FTranspose) )    
    

    FNP = FTranspose.transpose()
    return FNP

fractionsNP = solveForFractions( signatureDF, mixtureDF )
print(f'\n******************* fractionsNP.shape : {fractionsNP.shape}\n')
fractionsNP

 sigNP.shape : (15, 2) mixtureNP.shape : (15, 8)




******************* fractionsNP.shape : (8, 2)



array([[1.00000000e+00, 9.24459074e-17],
       [4.43320643e-17, 1.00000000e+00],
       [9.00000000e-01, 1.00000000e-01],
       [9.90000000e-01, 1.00000000e-02],
       [9.99000000e-01, 1.00000000e-03],
       [9.99900000e-01, 1.00000000e-04],
       [9.99990000e-01, 1.00000000e-05],
       [9.99999000e-01, 1.00000000e-06]])

In [6]:

FDF = pd.DataFrame(
    fractionsNP,
    index = mixtureDF.columns, 
    columns = ["UD", "Control", ]
)

FDF

,UD,Control
undiluted,1.000000e+00,9.244591e-17
control,4.433206e-17,1.000000e+00
inSilco_0.1,9.000000e-01,1.000000e-01
inSilco_0.01,9.900000e-01,1.000000e-02
inSilco_0.001,9.990000e-01,1.000000e-03
inSilco_0.0001,9.999000e-01,1.000000e-04
inSilco_1e-05,9.999900e-01,1.000000e-05
inSilco_1e-06,9.999990e-01,1.000000e-06


In [7]:
byRow = 1
FDF['rowSum'] =FDF.loc[:, ['UD', 'Control']].sum(axis=byRow)
FDF

,UD,Control,rowSum
undiluted,1.000000e+00,9.244591e-17,1.0
control,4.433206e-17,1.000000e+00,1.0
inSilco_0.1,9.000000e-01,1.000000e-01,1.0
inSilco_0.01,9.900000e-01,1.000000e-02,1.0
inSilco_0.001,9.990000e-01,1.000000e-03,1.0
inSilco_0.0001,9.999000e-01,1.000000e-04,1.0
inSilco_1e-05,9.999900e-01,1.000000e-05,1.0
inSilco_1e-06,9.999990e-01,1.000000e-06,1.0


In [11]:
FDF['%UD'] =  (FDF['UD'] / FDF['rowSum'] * 100.0).round(decimals=3) 
# print()
# print( FDF['Control'] / FDF['rowSum'] * 100.0)
FDF['%Control'] =  (FDF['Control'] / FDF['rowSum'] * 100.0).round(decimals=4) 
FDF

,UD,Control,rowSum,%UD,%Control
undiluted,1.000000e+00,9.244591e-17,1.0,100.000,0.0000
control,4.433206e-17,1.000000e+00,1.0,0.000,100.0000
inSilco_0.1,9.000000e-01,1.000000e-01,1.0,90.000,10.0000
inSilco_0.01,9.900000e-01,1.000000e-02,1.0,99.000,1.0000
inSilco_0.001,9.990000e-01,1.000000e-03,1.0,99.900,0.1000
inSilco_0.0001,9.999000e-01,1.000000e-04,1.0,99.990,0.0100
inSilco_1e-05,9.999900e-01,1.000000e-05,1.0,99.999,0.0010
inSilco_1e-06,9.999990e-01,1.000000e-06,1.0,100.000,0.0001
